In [1]:
!pip install onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [onnxscript]3 [onnxscript]


In [2]:
import math
import torch
import json
import os
import onnx
import onnxruntime
import shutil
import onnxscript

from collections import Counter
from tokenizers import Tokenizer

In [3]:
os.makedirs('/kaggle/working/ref/', exist_ok=True)

In [4]:
path = '/kaggle/working/ref/'
classic_model_path = '/kaggle/input/datasets/absolute1000/classic-model-after-sft/SFT_BEST.pt'
tokenizer = '/kaggle/input/datasets/absolute1000/tokenized-llm-training/tokenizer.json'
device = 'gpu' if torch.cuda.is_available() else 'cpu'
config_classic = {
    "embedding_size": 384,
    "encoder_num": 6,
    "heads_encoder": 6,
    "decoder_num": 2,
    "heads_decoder": 6,
}
tok = Tokenizer.from_file(tokenizer)
PAD_ID = tok.encode('<pad>').ids[0]
MASK_ID = tok.encode('<mask>').ids[0]
USER_ID = tok.encode('<user>').ids[0]
BOT_ID = tok.encode('<bot>').ids[0]
EOS_ID = tok.encode('<eos>').ids[0]
print(PAD_ID, MASK_ID, USER_ID, BOT_ID, EOS_ID)

0 1 2 3 4


In [5]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN_Classic(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [6]:
model = NN_Classic(**config_classic)
ckpt = torch.load(classic_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'] )
model.eval()
sum(p.numel() for p in model.parameters())

20061696

In [7]:
def ngram(seq, n):
    nxt, prv = dict(), dict()

    for i in range(0, len(seq) - n + 1):
        phrase_to_right = seq[i:i + n - 1]
        phrase_to_left = seq[i + 1:i + n]
        to_right_target = seq[i + n - 1]
        to_left_target = seq[i]
        if not(PAD_ID in phrase_to_right or PAD_ID == to_right_target):
            nxt.setdefault(tuple(phrase_to_right), set()).add(to_right_target)
        if not(PAD_ID in phrase_to_left or PAD_ID == to_left_target):
            prv.setdefault(tuple(phrase_to_left), set()).add(to_left_target)
    
    return nxt, prv

def penalize_ngram(ngram_end, ngram_start, ngram_len, ans_candidate_final, candidate_eos_idx, ans_candidate_logits, ngram_penalty, ANSWER_LEN):

    for i in range(candidate_eos_idx - ngram_len + 1):
        substring_for_ngram_to_end = ans_candidate_final[0, i:i + ngram_len - 1].tolist()
        substring_for_ngram_to_start = ans_candidate_final[0, i + 1:i + ngram_len].tolist()

        end_cands_to_penalize = ngram_end.get(tuple(substring_for_ngram_to_end), None)
        start_cands_to_penalize = ngram_start.get(tuple(substring_for_ngram_to_start), None)

        if end_cands_to_penalize is not None and MASK_ID not in substring_for_ngram_to_end:
            for word in end_cands_to_penalize:
                ans_candidate_logits[:, i + ngram_len - 1, word] -= ngram_penalty

        if start_cands_to_penalize is not None and MASK_ID not in substring_for_ngram_to_start:
            for word in start_cands_to_penalize:
                ans_candidate_logits[:, i, word] -= ngram_penalty

def gstar(device, model, margin_for_gstar_replace, start_block_idx, end_block_idx, KV_cache, KV_mask,
          ans_candidate_final, context_ids, rounds, is_global=False):
    block_size = end_block_idx - start_block_idx
    indexer = torch.arange(start_block_idx, end_block_idx, device=device)  # [k] — позиции в ответе
    rows = torch.arange(block_size, device=device)                         # [k] — номера копий

    # last user phrase between last <user> and last <bot>
    context = context_ids[0].tolist()
    last_user = len(context) - 1 - context[::-1].index(USER_ID) if USER_ID in context else -1
    last_bot = len(context) - 1 - context[::-1].index(BOT_ID)
    last_user_phrase = set(context[last_user + 1:last_bot])

    def leave_one_out_probs():
        batch = ans_candidate_final.repeat(block_size, 1)            # [k, 64] — k answer copies
        batch[rows, indexer] = MASK_ID                               # pos indexer[r] masked in r_i copy
        temp_logits = model.decode(batch, KV_cache.expand(block_size, -1, -1),
                                   KV_mask.expand(block_size, -1, -1)).float()   # [k, 64, V]
        temp_logits = temp_logits[rows, indexer]                     # [k, V] only masked pos
        temp_logits[:, (MASK_ID, PAD_ID, EOS_ID)] = -float('inf')
        return torch.softmax(temp_logits, dim=-1)

    for step in range(rounds):
        temp_probs = leave_one_out_probs()
        cur_token = ans_candidate_final[0, indexer]                  # [k]
        p_cur = temp_probs[rows, cur_token]                          # [k]
        p_best, best_token = temp_probs.max(dim=-1)                  # [k], [k]

        token_to_replace, pr, diff = None, None, -float('inf')
        for pretender in range(block_size):
            is_bad = p_cur[pretender] < margin_for_gstar_replace * p_best[pretender]
            is_other = best_token[pretender] != cur_token[pretender]
            is_free = cur_token[pretender].item() not in last_user_phrase
            gap = (p_best[pretender] - p_cur[pretender]).item()
            if is_bad and is_other and is_free and gap > diff:
                token_to_replace, pr, diff = best_token[pretender], pretender, gap

        if token_to_replace is None:
            break
        ans_candidate_final[0, indexer[pr]] = token_to_replace

    if is_global:
        temp_probs = leave_one_out_probs()
        p_cur = temp_probs[rows, ans_candidate_final[0, indexer]]
        return p_cur.clamp_min(1e-9).log().mean().item()

@torch.no_grad()
def generate(model, 
             context_ids, 
             mean_length_scale, 
             min_len, 
             n_candidates, 
             block_len, 
             repeat_penalty, 
             ngram_ans_len, 
             ngram_user_len, 
             ngram_penalty, 
             temperature,
             top_k,
             margin_for_gstar_replace,
             ANSWER_LEN,
             block_rounds,
             global_rounds,
             greedy,
             device
            ):
    context_ids = context_ids[None, :] # [256] -> [1, 256]
    KV_cache, KV_mask = model.encode(context_ids) # [1, 256, 384], [1, 1, 384]
    #answer_logits = torch.ones(size=(1, 64), dtype=torch.long) [1, 64]
    answer_probs = torch.nn.functional.softmax(
        model.decode(torch.full((1, ANSWER_LEN), MASK_ID, dtype=torch.long, device=device), KV_cache, KV_mask).float(), dim=2) # [1, 64, 12288]
    
    Len_mean = ((1 - answer_probs[..., EOS_ID] - answer_probs[..., PAD_ID]).sum(dim=1) * mean_length_scale).item()
    length_scale_candidates = torch.linspace(torch.tensor(0.8), torch.tensor(1.2), n_candidates) if n_candidates > 1 else torch.tensor([1.0])
    Len_candidates = ((Len_mean * length_scale_candidates).round()).clamp(min_len, 63).long() # [n_candidates]

    ngram_end_user, ngram_start_user = ngram(context_ids[0].tolist(), ngram_user_len)

    best_candidate, best_candidate_score = None, -float('inf')
    for idx in range(n_candidates):
        ans_candidate_final = torch.full((1, ANSWER_LEN), MASK_ID, dtype=torch.long, device=device)
        ans_candidate_final[:, Len_candidates[idx]] = EOS_ID
        ans_candidate_final[:, Len_candidates[idx] + 1:] = PAD_ID

        for start_block_idx in range(0, Len_candidates[idx].item(), block_len):
            end_block_idx = min(start_block_idx + block_len, Len_candidates[idx].item())

            block_region = torch.zeros(size=(64,), device=device).bool()
            block_region[start_block_idx:end_block_idx] = True

            while True:
                masks_still_to_open = (ans_candidate_final[0] == MASK_ID) & block_region
                if not masks_still_to_open.any().item(): break

                ans_candidate_logits = model.decode(ans_candidate_final, KV_cache, KV_mask).float()

                #penalty wrong end
                ans_candidate_logits[..., (MASK_ID, PAD_ID, EOS_ID)] = -float('inf')

                #penalty repeat
                used = ans_candidate_final[0][ans_candidate_final[0] != MASK_ID].unique()
                lu = ans_candidate_logits[0, :, used]
                ans_candidate_logits[0, :, used] = torch.where(lu > 0, lu / repeat_penalty, lu * repeat_penalty)

                #penalty ngram
                ngrams_ans_end, ngrams_ans_start = ngram(ans_candidate_final[0].tolist(), ngram_ans_len)
                penalize_ngram(ngrams_ans_end, ngrams_ans_start, ngram_ans_len, ans_candidate_final, Len_candidates[idx].item(), 
                               ans_candidate_logits, ngram_penalty, ANSWER_LEN)
                penalize_ngram(ngram_end_user, ngram_start_user, ngram_user_len, ans_candidate_final, Len_candidates[idx].item(), 
                               ans_candidate_logits, ngram_penalty, ANSWER_LEN)

                # temperature
                ans_candidate_logits[0] /= temperature

                #top k
                min_logit_per_row = ans_candidate_logits.topk(top_k, dim=2).values[:, :, -1:] # [64, 1]
                ans_candidate_logits = ans_candidate_logits.masked_fill(ans_candidate_logits < min_logit_per_row, -float("inf"))

                # probs and word choosing
                ans_candidate_probs = torch.softmax(ans_candidate_logits, 2).squeeze(0) # [1, 64, 12288] -> [64, 12288]

                if greedy:
                    tokens_choice = torch.argmax(ans_candidate_probs, dim=1, keepdim=True)
                else:
                    tokens_choice = torch.multinomial(ans_candidate_probs, 1) # [64, 1]
                confidence = ans_candidate_probs.gather(-1, tokens_choice).squeeze(1) # [64, 1] -> [64]

                # open only one mask per step
                i = confidence.masked_fill(~masks_still_to_open, -1).argmax().item()
                ans_candidate_final[0, i] = tokens_choice[i, 0]

            # gstar
            gstar(device, model, margin_for_gstar_replace, start_block_idx, end_block_idx, KV_cache, KV_mask, ans_candidate_final, context_ids, 
                  rounds=block_rounds)
        this_candidate_score = gstar(device, model, margin_for_gstar_replace, 0, Len_candidates[idx].item(), KV_cache, KV_mask, ans_candidate_final, 
                                     context_ids, rounds=global_rounds, is_global=True)
        if this_candidate_score > best_candidate_score:
            best_candidate_score = this_candidate_score
            best_candidate = ans_candidate_final
            
    return best_candidate.squeeze(0)

In [8]:
SAMPLER_SFT_CLASSIC = {
    "mean_length_scale": 1.0,
    "min_len": 3,
    "n_candidates": 4,
    "block_len": 8,
    "repeat_penalty": 1.05,
    "ngram_ans_len": 3,
    "ngram_user_len": 4,
    "ngram_penalty": 5.0,
    "temperature": 0.5,
    "top_k": 50,
    "margin_for_gstar_replace": 0.5,
    "ANSWER_LEN": 64,
    "block_rounds": 2,
    "global_rounds": 2,
    'greedy': False
}

In [10]:
ids = [USER_ID] + tok.encode('Hello there my dear friend! How are you?').ids + [BOT_ID]
out = generate(model, context_ids=torch.tensor(ids, device='cpu'), device='cpu', **SAMPLER_SFT_CLASSIC).tolist()
print(tok.decode(out[:out.index(EOS_ID)]))

Hi, I am good how are you doing?


In [11]:
GOLDEN_GEN = [
    'Hello there my dear friend! How are you?',
    'I have just bought a new puppy',
    'What is the kilometer?',
    'Such a nice weather',
    'Such a terrible weather',
    'What is his abilities?',
    'I got a new computer',
    "Let's go play football after dinner",
    'I will do my homework whole day',
    'I want to chat with you'
]

In [12]:
with open(path + 'golden_gen.json', 'w', encoding='UTF-8') as f:
    l = []
    for prompt in GOLDEN_GEN:
        prompt = tok.encode(prompt).ids
        prompt = [PAD_ID] * (256 - 2 - len(prompt)) + [USER_ID] + prompt + [BOT_ID]
        prompt = torch.tensor(prompt, device='cpu')
        answer = generate(model, context_ids=prompt, device='cpu', **{**SAMPLER_SFT_CLASSIC, 'greedy': True}).tolist()
        
        stop_id = answer.index(EOS_ID)
        history_ids = prompt.tolist()
        reply_ids = answer
        reply_text = tok.decode(answer[:stop_id])
        l.append({'history_ids': history_ids, 'reply_ids':reply_ids, 'reply_text':reply_text})
    json.dump(l, f)

In [13]:
class EncoderExport(torch.nn.Module):

    def __init__(self, model):
        super().__init__()
        self.m = model

    def forward(self, context_ids):
        return self.m.encode(context_ids) # KV_cache, KV_mask

class DecoderExport(torch.nn.Module):

    def __init__(self, model):
        super().__init__()
        self.m = model

    def forward(self, KV_cache, KV_mask, answer_ids):
        KV_cache = KV_cache.expand(answer_ids.shape[0], -1, -1)
        KV_mask = KV_mask.expand(answer_ids.shape[0], -1, -1)
        return self.m.decode(answer_ids, KV_cache, KV_mask)

In [14]:
enc = EncoderExport(model)
dec = DecoderExport(model)

prompt = 'I will do my homework whole day'
prompt = tok.encode(prompt).ids
prompt = [PAD_ID] * (256 - 2 - len(prompt)) + [USER_ID] + prompt + [BOT_ID]
prompt = torch.tensor(prompt, device='cpu').unsqueeze(0)

enc.eval(), dec.eval()
with torch.no_grad():
    memory, mask = enc.forward(prompt)
    ans_ids = torch.full((2, 64), MASK_ID, dtype=torch.long, device='cpu')
    ans = dec.forward(memory, mask, ans_ids)

    torch.onnx.export(
        model=enc, args=(prompt,), f=path + 'encoder.onnx', export_params=True, opset_version=18, input_names=['context_ids'], output_names=['memory', 'mask'],
        dynamic_axes=None, verbose=False, dynamo=True, external_data=False)

/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [24]:
with torch.no_grad():
    torch.onnx.export(
        model=dec, args=(memory, mask, ans_ids), f=path + 'decoder.onnx', export_params=True, opset_version=18, input_names=['memory', 'mask', 'answer_ids'], output_names=['logits'],
        dynamic_shapes=(None, None, {0: 'batch'}), verbose=False, dynamo=True, external_data=False)

/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [25]:
enc_onnx = onnx.load(path + 'encoder.onnx')
dec_onnx = onnx.load(path + 'decoder.onnx')

In [26]:
onnx.checker.check_model(enc_onnx)

In [27]:
onnx.checker.check_model(dec_onnx)

In [28]:
Counter(n.op_type for n in enc_onnx.graph.node)

Counter({'Equal': 1,
         'Not': 2,
         'Unsqueeze': 145,
         'Gather': 1,
         'Pow': 13,
         'ReduceMean': 13,
         'Add': 97,
         'Sqrt': 13,
         'Reciprocal': 13,
         'Mul': 326,
         'MatMul': 204,
         'Slice': 144,
         'Sub': 72,
         'Concat': 78,
         'Reshape': 72,
         'Transpose': 36,
         'Where': 36,
         'Div': 36,
         'Softmax': 36,
         'Sigmoid': 6})

In [29]:
Counter(n.op_type for n in dec_onnx.graph.node)

Counter({'Shape': 1,
         'Concat': 29,
         'Expand': 2,
         'Gather': 1,
         'Pow': 7,
         'ReduceMean': 7,
         'Add': 37,
         'Sqrt': 7,
         'Reciprocal': 7,
         'Mul': 114,
         'MatMul': 131,
         'Slice': 48,
         'Sub': 24,
         'Unsqueeze': 48,
         'Reshape': 24,
         'Transpose': 25,
         'Div': 24,
         'Softmax': 24,
         'Not': 1,
         'Where': 12,
         'Sigmoid': 2})

In [30]:
print(os.path.getsize(path + 'encoder.onnx'), os.path.getsize(path + 'decoder.onnx'))

68315807 39157142


In [31]:
class OrtBackend():

    def __init__(self, enc_path, dec_path):
        self.encoder = onnxruntime.InferenceSession(enc_path, providers=['CPUExecutionProvider'])
        self.decoder = onnxruntime.InferenceSession(dec_path, providers=['CPUExecutionProvider'])

    def encode(self, context_ids):
        # context_ids [1, 256] torch.tensor
        memory, mask = self.encoder.run(None, {'context_ids': context_ids.numpy()})
        return torch.from_numpy(memory), torch.from_numpy(mask)

    def decode(self, answer_ids, memory, mask):
        # 
        return torch.from_numpy(self.decoder.run(None, {'memory': memory[:1].numpy(), 'mask': mask[:1].numpy(), 'answer_ids': answer_ids.numpy()})[0])

In [32]:
with open(path + 'golden_gen.json', 'r', encoding='UTF-8') as f:
    equal = 0
    data = json.load(f)
    model_check = OrtBackend(path + 'encoder.onnx', path + 'decoder.onnx')
    for g in data:
        history_ids, reply_ids, reply_text = g['history_ids'], g['reply_ids'], g['reply_text']
        history_ids = torch.tensor(history_ids, device='cpu')
        answer = generate(model_check, context_ids=history_ids, device='cpu', **{**SAMPLER_SFT_CLASSIC, 'greedy': True}).tolist()

        if answer == g['reply_ids']:
            equal += 1
            continue
        print(f"TRUE: {repr(reply_text)}")
        print(f"AFTER ONNX: {repr(tok.decode(answer))}")
    print("Equal", equal)

Equal 10


In [33]:
MODEL_INFERENCE_CONFIG = {
    'PAD_ID': PAD_ID,
    'MASK_ID': MASK_ID,
    'USER_ID' : USER_ID,
    'BOT_ID': BOT_ID,
    'EOS_ID': EOS_ID,
    'CONTEXT_LEN': 256,
    'ANSWER_LEN': 64,
    'PAD_SIDE': "left",
    'SAMPLER_SETTINGS': SAMPLER_SFT_CLASSIC,
    'MODEL_SETTINGS': config_classic
}

In [34]:
shutil.copy(tokenizer, path)

'/kaggle/working/ref/tokenizer.json'

In [35]:
with open(path + 'config.json', 'w', encoding='UTF-8') as f:
    json.dump(MODEL_INFERENCE_CONFIG, f)

In [36]:
shutil.make_archive('/kaggle/working/ref', 'zip', path)

'/kaggle/working/ref.zip'